# К16 · A3 · Baseline для математических решений

Запустите все ячейки в Jupyter или Google Colab. Достаточно стандартной библиотеки Python 3.10 или выше. Поместите три приложенных файла в одну папку с ноутбуком или в папку `audit/`.

| Файл | Как используется |
| --- | --- |
| `slice(3).jsonl` | 52 задачи и проверяемые решения |
| `annotations(3).csv` | 28 человеческих меток A3 для расчёта предварительного Error Rate |
| `baseline_predictions(2).csv` | Шаблон, задающий порядок строк и ID |

Исходные файлы сохраняются. Ноутбук записывает готовый `baseline_predictions.csv` в папку с данными. Также поддерживаются входные имена `slice.jsonl` и `annotations.csv`.

In [1]:
from pathlib import Path

# None — искать файлы в текущей папке, audit/, upload/ и ../audit/.
# Если файлы лежат в другом месте, задайте путь, например Path("/content").
DATA_DIR = None


def resolve_file(directory, names):
    return next((directory / name for name in names if (directory / name).is_file()), None)


directories = (
    (Path(DATA_DIR),)
    if DATA_DIR is not None
    else (Path("."), Path("audit"), Path("upload"), Path("../audit"))
)
for directory in directories:
    slice_path = resolve_file(directory, ("slice(3).jsonl", "slice.jsonl", "slice(2).jsonl"))
    annotations_path = resolve_file(
        directory, ("annotations(3).csv", "annotations.csv", "annotations(2).csv")
    )
    if slice_path is not None and annotations_path is not None:
        audit_dir = directory.resolve()
        break
else:
    raise FileNotFoundError(
        "Поместите slice(3).jsonl и annotations(3).csv в одну папку и укажите её в DATA_DIR."
    )

template_path = resolve_file(
    audit_dir, ("baseline_predictions(2).csv", "baseline_predictions.csv",
                "baseline_predictions(3).csv")
)
baseline_path = audit_dir / "baseline_predictions.csv"

## 1. Извлечение и сравнение ответов

Приоритет у последнего `\boxed{…}` или `\fbox{…}`. Для ответов без рамки используются явно указанный ответ и финальное математическое выражение. Поддерживаемые выражения приводятся к точным рациональным числам. Например, `1/2`, `0,5` и `50%` равны. Запись `1,000` читается как 1000 по правилам англоязычного исходника. Нераспознанные ответы получают вердикт `incorrect` и отдельно перечисляются.

Baseline оценивает только финальный ответ. Поэтому решение с ошибкой в рассуждении и верным итогом может получить `correct`.

In [2]:
import ast
import csv
import json
import operator
import re
from fractions import Fraction
from math import comb, factorial, isqrt

LABELS = {"correct", "incorrect"}
OPERATIONS = {ast.Add: operator.add, ast.Sub: operator.sub,
              ast.Mult: operator.mul, ast.Div: operator.truediv}


def extract_final_answer(solution):
    """Use the last boxed answer, then an explicit answer or final math span."""
    solution = re.sub(r"\[asy\].*?\[/asy\]", "", solution, flags=re.S).strip()
    boxes = []
    for match in re.finditer(r"\\(?:boxed|fbox)\s*\{", solution):
        depth = 1
        for pos in range(match.end(), len(solution)):
            depth += (solution[pos] == "{") - (solution[pos] == "}")
            if depth == 0:
                boxes.append(solution[match.end():pos])
                break
    if boxes:
        return boxes[-1].strip()

    explicit = re.search(
        r"\b(?:final\s+answer|(?:the\s+)?answer)\s*(?::|is\b)\s*(.+)\Z",
        solution, flags=re.I
    )
    if explicit:
        solution = explicit[1].strip()
    math_span = re.search(
        r"(?:\$\$([^$]+)\$\$|\$([^$]+)\$|\\\((.*?)\\\)|\\\[(.*?)\\\])\s*[.!]?\s*\Z",
        solution, flags=re.S
    )
    if math_span:
        solution = next(group for group in math_span.groups() if group is not None)
    elif not explicit:
        solution = solution.splitlines()[-1] if solution else ""
    solution = solution.strip().rstrip(".")
    if "=" in solution:
        solution = solution.rsplit("=", 1)[1].strip()
    return solution or None


def translate_expression(answer):
    """Convert supported LaTeX and numerical notation into an arithmetic expression."""
    answer = answer.strip().replace("−", "-").replace("×", "*").replace("÷", "/")
    answer = re.sub(r"\\(?:left|right|displaystyle)\b|\\[,!;: ]", "", answer)
    for before, after in ((r"\cdot", "*"), (r"\times", "*"), (r"\div", "/"), (r"\%", "%")):
        answer = answer.replace(before, after)
    answer = re.sub(r"(?<=\d)[ \u00a0\u202f](?=\d{3}\b)", "", answer)
    answer = re.sub(r"\b[1-9]\d{0,2}(?:,\d{3})+\b", lambda m: m[0].replace(",", ""), answer)
    # Normalize decimal commas before generating function calls.
    answer = re.sub(r"(?<=\d),(?=\d)", ".", answer)
    replacements = (
        (r"\\[dt]?frac\s*\{([^{}]+)\}\s*\{([^{}]+)\}", r"((\1)/(\2))"),
        (r"\\sqrt\s*\{([^{}]+)\}", r"sqrt((\1))"),
        (r"\\[dt]?binom\s*\{([^{}]+)\}\s*\{([^{}]+)\}", r"comb((\1),(\2))"),
    )
    for _ in range(20):
        previous = answer
        for pattern, replacement in replacements:
            answer = re.sub(pattern, replacement, answer)
        answer = re.sub(r"\^\{([^{}]+)\}", r"^(\1)", answer)
        if answer == previous:
            break
    answer = re.sub(r"(?<![\w.])(\d+)!", r"factorial(\1)", answer)
    answer = answer.replace("^", "**")
    if answer.endswith("%"):
        answer = f"({answer[:-1]})/100"
    return answer.strip()


def normalize_final_answer(answer):
    """Return an exact rational value, or None for unsupported expressions."""
    if not answer or len(answer) > 300:
        return None
    source = translate_expression(answer)

    def read(node):
        if isinstance(node, ast.Constant) and type(node.value) in (int, float):
            literal = ast.get_source_segment(source, node)
            if not re.fullmatch(r"(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][+-]?\d{1,2})?", literal):
                raise ValueError("Unsupported number")
            return Fraction(literal)
        if isinstance(node, ast.UnaryOp) and type(node.op) in (ast.UAdd, ast.USub):
            value = read(node.operand)
            return -value if isinstance(node.op, ast.USub) else value
        if isinstance(node, ast.BinOp):
            left, right = read(node.left), read(node.right)
            if type(node.op) in OPERATIONS:
                return OPERATIONS[type(node.op)](left, right)
            if isinstance(node.op, ast.Pow) and right.denominator == 1 and abs(right) <= 50:
                if max(left.numerator.bit_length(), left.denominator.bit_length()) * abs(right) <= 50_000:
                    return left ** int(right)
        if isinstance(node, ast.Call) and isinstance(node.func, ast.Name) and not node.keywords:
            args = [read(arg) for arg in node.args]
            if node.func.id == "sqrt" and len(args) == 1 and args[0] >= 0:
                p, q = args[0].as_integer_ratio()
                if isqrt(p)**2 == p and isqrt(q)**2 == q:
                    return Fraction(isqrt(p), isqrt(q))
            if all(a.denominator == 1 and 0 <= a <= 100 for a in args):
                if node.func.id == "factorial" and len(args) == 1:
                    return Fraction(factorial(int(args[0])))
                if node.func.id == "comb" and len(args) == 2:
                    return Fraction(comb(*map(int, args)))
        raise ValueError("Unsupported expression")

    try:
        return read(ast.parse(source, mode="eval").body)
    except (SyntaxError, ValueError, ArithmeticError, RecursionError):
        return None


def compare_final_answers(candidate_solution, reference_solution):
    """Return the binary verdict and whether both answers could be normalized."""
    candidate = normalize_final_answer(extract_final_answer(candidate_solution))
    reference = normalize_final_answer(extract_final_answer(reference_solution))
    if reference is None:
        raise ValueError("Reference answer could not be normalized")
    return ("correct" if candidate == reference else "incorrect"), candidate is not None

## 2. Эталонные ответы

Эталонные значения для текущей выборки вычисляются из условий. Функция получает ответы отдельно от текста проверяемого решения. Эталон финального ответа и эталонная метка корректности всего решения используются на разных этапах.

In [3]:
from datetime import date
from fractions import Fraction as F
from itertools import combinations, product
from math import ceil, floor, prod


def reference_solutions():
    # A painted unit cube lies on a boundary along one axis and at the
    # midpoint along at least one of the other axes.
    painted = sum(
        any(position[i] in (0, 4) and
            any(position[j] == 2 for j in range(3) if j != i)
            for i in range(3))
        for position in product(range(5), repeat=3)
    )

    # Each of five green marbles can contribute at most two color changes.
    # Equality requires all ten changes, with a red marble in each of six gaps.
    marble_arrangements = comb(10 + 5, 5)

    cards = list(product(range(1, 14), range(4)))
    fours = [card for card in cards if card[0] == 4]
    clubs = [card for card in cards if card[1] == 0]
    twos = [card for card in cards if card[0] == 2]
    valid_deals = sum(len(set(deal)) == 3 for deal in product(fours, clubs, twos))

    factorion, = [n for n in range(100, 1000)
                  if n == sum(factorial(int(digit)) for digit in str(n))]
    coin_count, = {sum(counts) for counts in product(range(11), repeat=4)
                   if sum(n * t for n, t in zip(counts, (155, 195, 135, 175))) == 1400}
    pascal_rows = sum(all(comb(n, k) % 2 == 0 for k in range(1, n))
                      for n in range(2, 20))

    answers = {
        "0453": 26**4 * 10**2 - 26**3 * 10**3,
        "0509": 5**3 - painted,
        "0231": F(sum(comb(8, k) for k in range(5, 9)), 2**8),
        "0064": (date(2005, 1, 1) - date(2001, 1, 1)).days,
        "0491": sum(a <= 32 - a for a in range(1, 32)),
        "0464": comb(16, 5) - comb(9, 5),
        "0711": comb(5, 1),
        "0584": F(prod((13, 12, 11)), prod((52, 51, 50))),
        "0392": sum("0" not in str(a) + str(1000 - a) for a in range(1, 1000)),
        "0215": ceil(F("10.3")**3) - floor(F("10.2")**3) - 1,
        "0699": sum(comb(8, k) for k in range(9)),
        "0214": 2 * 10 * 9,
        "0244": sum(a < b and c < b
                    for a, b, c in product(range(1, 10), range(10), range(10))),
        "0408": marble_arrangements % 1000,
        "0065": F(sum(a * b % 2 == 0
                      for a, b in product(range(1, 5), range(1, 4))), 12),
        "0135": F(factorial(9) * factorial(5) * factorial(2), factorial(8) * factorial(6)),
        "0585": 6 * 4 * 6 - 4,
        "0262": comb(25, 2),
        "0621": factorial(4) * factorial(3),
        "0641": sum(len(set(str(n))) == 3 for n in range(101, 1000, 2)),
        "0646": comb(19, 9),
        "0202": 2 * F(5, 8) * F(3, 8),
        "0630": F(sum(isqrt(a + b)**2 == a + b
                      for a, b in product(range(1, 7), repeat=2)), 36),
        "0702": sum(all(n % d for d in range(2, isqrt(n) + 1)) for n in range(2, 31)),
        "0557": F(sum((x * y - x - y) % 2 == 0
                      for x, y in combinations(range(1, 11), 2)), comb(10, 2)),
        "0374": comb(8, 4),
        "0289": F((4 + 1) * 3, 2 * 3 * 6),
        "0760": sum(str(n) == str(n)[::-1] for n in range(101, 500)),
        "0587": F(valid_deals, 52 * 51 * 50),
        "0462": 3 + 2 * (15 - 1),
        "0556": 3 * sum(range(1, 7)) - sum((1, 1, 2, 3, 4, 5, 6)),
        "0594": 1 - F(1, 3) - F(5, 12),
        "0146": factorion,
        "0084": 2 * F(1, 6),
        "0669": (43 + 5) * 3 // 4,
        "0765": pascal_rows,
        "0472": F(2, comb(8, 4)),
        "0196": F(factorial(6) + factorial(7), factorial(5)),
        "0604": round(10 * F(1, 6) * F(5, 6)**9, 3),  # Problem requests thousandths.
        "0608": 3 * F(1, 3) - 2 * F(2, 3),
        "0382": sum(len(set(str(n))) <= 2 for n in range(1, 10000)),
        "0132": 1 - F(4, 7),
        "0432": factorial(8) // 24,  # Eight faces and three rotations per face.
        "0201": sum(n // 10 > n % 10 for n in range(10, 100)),
        "0501": F(factorial(7), 5 * 8 * 2),
        "0200": F(1, comb(5, 3)),
        "0468": F(1, 2)**6,
        "0538": 3 * factorial(4) * 2**4,  # Center 1, 5 or 9, then four opposite pairs.
        "0660": coin_count,
        "0498": F(comb(3, 2), 2**3),
        "0577": comb(7, 2),
        "0572": factorial(3),
    }
    return {f"K16_MATH_{number}": rf"\boxed{{{answer}}}" for number, answer in answers.items()}

## 3. Проверка разметки A3 и расчёт baseline

Проверяем назначения A3, затем сравниваем финальные ответы на всех 52 задачах. Ячейка записывает `baseline_predictions.csv` в папку с данными. При наличии шаблона сохраняется порядок его строк.

In [4]:
def load_jsonl(path):
    with path.open(encoding="utf-8-sig") as stream:
        return [json.loads(line) for line in stream if line.strip()]


def load_csv(path):
    with path.open(encoding="utf-8-sig", newline="") as stream:
        return list(csv.DictReader(stream))


def index_labels(rows, column, description):
    indexed = {}
    for row in rows:
        item_id = row["id"]
        label = row[column]
        if not item_id or item_id in indexed or label not in LABELS:
            raise ValueError(f"{description}: invalid or duplicate label for {item_id!r}: {label!r}")
        indexed[item_id] = label
    return indexed


def require_same_ids(expected, observed, description):
    missing = sorted(expected.keys() - observed.keys())
    extra = sorted(observed.keys() - expected.keys())
    if missing or extra:
        raise ValueError(f"{description}: missing IDs {missing}, extra IDs {extra}")

In [5]:
rows = load_jsonl(slice_path)
candidates = {row["id"]: row["solution"] for row in rows}
if len(rows) != 52 or len(candidates) != len(rows):
    raise ValueError("В текущей выборке должно быть 52 задачи с уникальными ID.")

groups = {row["id"]: row["group"] for row in rows}
a3_rows = [row for row in load_csv(annotations_path) if row["annotator"] == "A3"]
a3_labels = index_labels(a3_rows, "label", "Разметка A3")
if len(a3_rows) != 28 or sum(groups.get(item_id) == "core" for item_id in a3_labels) != 20:
    raise ValueError("Для A3 нужны 20 общих и 8 дополнительных задач.")
if sum(groups.get(item_id) == "second" for item_id in a3_labels) != 8:
    raise ValueError("Проверьте ID дополнительных задач A3.")
for row in a3_rows:
    elapsed = Fraction(row["elapsed_seconds"])
    if elapsed <= 0:
        raise ValueError(f"Время проверки должно быть положительным для {row['id']}.")

references = reference_solutions()
require_same_ids(candidates, references, "Эталонные ответы")

order = list(candidates)
if template_path is not None:
    order = [row["id"] for row in load_csv(template_path)]
    if len(order) != len(set(order)):
        raise ValueError("В шаблоне baseline есть повторяющиеся ID.")
    require_same_ids(candidates, dict.fromkeys(order), "Шаблон baseline")

predicted = {}
unparsed = []
for item_id in order:
    verdict, parsed = compare_final_answers(candidates[item_id], references[item_id])
    predicted[item_id] = verdict
    if not parsed:
        unparsed.append(item_id)

with baseline_path.open("w", encoding="utf-8-sig", newline="") as stream:
    writer = csv.DictWriter(stream, fieldnames=("id", "verdict"))
    writer.writeheader()
    writer.writerows({"id": item_id, "verdict": verdict} for item_id, verdict in predicted.items())

print("Проверено строк A3: 28 (20 core + 8 second).")
print(f"Записан {baseline_path.name}: {len(predicted)} строк.")
print(f"Вердиктов correct: {sum(label == 'correct' for label in predicted.values())}.")
print(f"Нераспознанных финальных ответов: {len(unparsed)}.")
if unparsed:
    print("ID:", ", ".join(unparsed))

Проверено строк A3: 28 (20 core + 8 second).
Записан baseline_predictions.csv: 52 строк.
Вердиктов correct: 52.
Нераспознанных финальных ответов: 0.


## 4. Предварительный Error Rate

Для каждой из 28 задач A3 сравниваем вердикт baseline с меткой A3 из `annotations.csv`. Error Rate равен числу расхождений, делённому на 28. Это предварительная оценка по одному аннотатору.

In [6]:
def error_rate(predicted, gold):
    """Return the number of errors, sample size, and exact error rate."""
    require_same_ids(gold, predicted, "Evaluation")
    if not gold:
        raise ValueError("No examples to evaluate")
    errors = sum(predicted[item_id] != label for item_id, label in gold.items())
    return errors, len(gold), Fraction(errors, len(gold))


def report_rate(name, predicted, gold):
    errors, total, rate = error_rate(predicted, gold)
    print(f"{name}: {errors}/{total} ошибок ({float(rate):.2%})")
    return rate

In [7]:
a3_predictions = {item_id: predicted[item_id] for item_id in a3_labels}
report_rate("Baseline по A3, предварительно", a3_predictions, a3_labels)

Baseline по A3, предварительно: 0/28 ошибок (0.00%)
